# Уменьшение количества цветов изображения

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage import img_as_float
from sklearn.cluster import KMeans

In [4]:
original_photo = imread('parrots.jpg')
original_photo = img_as_float(original_photo)
img_width, img_height, channels = original_photo.shape


pixel_array = original_photo.reshape(-1, 3)

In [5]:
def compute_psnr(reference, compressed):
    error = np.mean((reference - compressed) ** 2)
    if error == 0:
        return 100
    return 10 * np.log10(1 / error)

In [6]:
def restore_from_clusters(assignments, centroids, strategy):
    result = np.zeros_like(pixel_array)
    
    for idx in range(len(centroids)):
        group_indices = assignments == idx
        group_data = pixel_array[group_indices]
        
        if strategy == 'mean':
            fill_value = group_data.mean(axis=0)
        elif strategy == 'median':
            fill_value = np.median(group_data, axis=0)
            
        result[group_indices] = fill_value
    
    return result.reshape(img_width, img_height, channels)

In [8]:
found_clusters = None
for k in range(1, 21):
    clustering_model = KMeans(n_clusters=k, init='k-means++', random_state=241)
    cluster_assignment = clustering_model.fit_predict(pixel_array)
    cluster_centers = clustering_model.cluster_centers_
    
    compressed_mean = restore_from_clusters(cluster_assignment, cluster_centers, 'mean')
    psnr_mean_value = compute_psnr(original_photo, compressed_mean)
    
    compressed_median = restore_from_clusters(cluster_assignment, cluster_centers, 'median')
    psnr_median_value = compute_psnr(original_photo, compressed_median)
    
    if psnr_mean_value > 20 or psnr_median_value > 20:
        found_clusters = k
        break

print(found_clusters)
with open('ans.txt', 'w') as file:
    file.write(str(found_clusters))

11
